# 13 - Ensembles y Gradient Boosting

Este notebook entrena y evalua XGBoost y LightGBM usando el mismo esquema de datos que el modelo final del Sprint 4.

Cambio clave: los modelos se entrenan con `data/processed/train_processed.csv` y se evaluan con `data/processed/test_processed.csv`, manteniendo el test real desbalanceado para que las metricas sean comparables con Random Forest.


In [10]:
import os
import time
import joblib
import warnings
from pathlib import Path
import lightgbm

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)
from scipy.stats import randint, uniform, loguniform
from xgboost import XGBClassifier

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ModuleNotFoundError:
    LGBMClassifier = None
    LIGHTGBM_AVAILABLE = False

warnings.filterwarnings("ignore")


In [11]:
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

TRAIN_PATH = ROOT / "data" / "processed" / "train_processed.csv"
TEST_PATH = ROOT / "data" / "processed" / "test_processed.csv"
MODELS_DIR = ROOT / "models"
RESULTS_PATH = MODELS_DIR / "ensemble_results_xgb_lgbm.csv"

XGB_BASE_PATH = MODELS_DIR / "xgboost_base.pkl"
LGBM_BASE_PATH = MODELS_DIR / "lightgbm_base.pkl"
XGB_TUNED_PATH = MODELS_DIR / "tuned_xgboost.pkl"
LGBM_TUNED_PATH = MODELS_DIR / "tuned_lightgbm.pkl"
RF_FINAL_PATH = MODELS_DIR / "best_tuned_model.pkl"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("TRAIN:", TRAIN_PATH)
print("TEST :", TEST_PATH)
print("LightGBM disponible:", LIGHTGBM_AVAILABLE)


ROOT: D:\_Trabajos\Desktop\Proyecto AD\dp261-g4
TRAIN: D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\data\processed\train_processed.csv
TEST : D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\data\processed\test_processed.csv
LightGBM disponible: True


In [12]:
TARGET_COL = "Revenue"

df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

if TARGET_COL not in df_train.columns or TARGET_COL not in df_test.columns:
    raise ValueError(f"La columna target '{TARGET_COL}' debe existir en train y test.")

X_train = df_train.drop(columns=[TARGET_COL])
y_train = df_train[TARGET_COL].astype(int)

X_test = df_test.drop(columns=[TARGET_COL])
y_test = df_test[TARGET_COL].astype(int)

if list(X_train.columns) != list(X_test.columns):
    missing_in_test = sorted(set(X_train.columns) - set(X_test.columns))
    missing_in_train = sorted(set(X_test.columns) - set(X_train.columns))
    raise ValueError(
        "Train y test no tienen las mismas columnas. "
        f"Faltan en test: {missing_in_test[:10]}; faltan en train: {missing_in_train[:10]}"
    )

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Distribucion train:")
print(y_train.value_counts().sort_index())
print((y_train.value_counts(normalize=True).sort_index() * 100).round(2))
print("Distribucion test real:")
print(y_test.value_counts().sort_index())
print((y_test.value_counts(normalize=True).sort_index() * 100).round(2))


X_train: (16476, 75)
X_test : (2441, 75)
Distribucion train:
Revenue
0    8238
1    8238
Name: count, dtype: int64
Revenue
0    50.0
1    50.0
Name: proportion, dtype: float64
Distribucion test real:
Revenue
0    2059
1     382
Name: count, dtype: int64
Revenue
0    84.35
1    15.65
Name: proportion, dtype: float64


## Funciones de evaluacion

La validacion cruzada se calcula sobre train para seleccionar hiperparametros. La comparacion final se hace sobre el test real, con `threshold=0.50` y `threshold=0.25` para reflejar dos politicas de negocio: precision conservadora vs mayor captura de compradores.


In [13]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
}

THRESHOLDS = [0.50, 0.25]


def evaluate_model_cv(model, X, y, cv, scoring, model_name, version):
    start_time = time.time()
    scores = cross_validate(
        estimator=model,
        X=X,
        y=y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )
    elapsed_time = time.time() - start_time

    return {
        "model": model_name,
        "version": version,
        "accuracy_cv_mean": scores["test_accuracy"].mean(),
        "accuracy_cv_std": scores["test_accuracy"].std(),
        "f1_cv_mean": scores["test_f1"].mean(),
        "f1_cv_std": scores["test_f1"].std(),
        "precision_cv_mean": scores["test_precision"].mean(),
        "precision_cv_std": scores["test_precision"].std(),
        "recall_cv_mean": scores["test_recall"].mean(),
        "recall_cv_std": scores["test_recall"].std(),
        "roc_auc_cv_mean": scores["test_roc_auc"].mean(),
        "roc_auc_cv_std": scores["test_roc_auc"].std(),
        "time_seconds": round(elapsed_time, 2),
    }


def evaluate_model_test(model, X_test, y_test, threshold=0.50):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    return {
        "threshold": threshold,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred, zero_division=0),
        "test_recall": recall_score(y_test, y_pred),
        "test_roc_auc": roc_auc_score(y_test, y_proba),
        "test_pr_auc": average_precision_score(y_test, y_proba),
        "test_predicted_positive": int(y_pred.sum()),
        "test_tn": int(tn),
        "test_fp": int(fp),
        "test_fn": int(fn),
        "test_tp": int(tp),
    }


def assert_feature_compatibility(model, X, model_name):
    if hasattr(model, "feature_names_in_"):
        expected = list(model.feature_names_in_)
        current = list(X.columns)
        if expected != current:
            raise ValueError(
                f"{model_name} no es compatible con estas columnas. "
                f"Esperaba {len(expected)} columnas y recibio {len(current)}."
            )


## Modelos base


In [14]:
results = []
trained_models = {}

xgb_base = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
)

xgb_base_result = evaluate_model_cv(
    model=xgb_base,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    model_name="XGBoost",
    version="base",
)

xgb_base.fit(X_train, y_train)
joblib.dump(xgb_base, XGB_BASE_PATH)
trained_models[("XGBoost", "base")] = xgb_base
results.append({**xgb_base_result, "best_params": "", "model_path": str(XGB_BASE_PATH)})

pd.DataFrame(results)


,model,version,accuracy_cv_mean,accuracy_cv_std,f1_cv_mean,f1_cv_std,precision_cv_mean,precision_cv_std,recall_cv_mean,recall_cv_std,roc_auc_cv_mean,roc_auc_cv_std,time_seconds,best_params,model_path
0,XGBoost,base,0.941854,0.003506,0.941711,0.003435,0.944133,0.004759,0.939306,0.002331,0.987294,0.001545,13.58,,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...


In [15]:
if LIGHTGBM_AVAILABLE:
    lgbm_base = LGBMClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1,
        n_jobs=-1,
    )

    lgbm_base_result = evaluate_model_cv(
        model=lgbm_base,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        model_name="LightGBM",
        version="base",
    )

    lgbm_base.fit(X_train, y_train)
    joblib.dump(lgbm_base, LGBM_BASE_PATH)
    trained_models[("LightGBM", "base")] = lgbm_base
    results.append({**lgbm_base_result, "best_params": "", "model_path": str(LGBM_BASE_PATH)})
else:
    print("LightGBM no esta instalado en este entorno. Se omite este bloque.")

pd.DataFrame(results)


,model,version,accuracy_cv_mean,accuracy_cv_std,f1_cv_mean,f1_cv_std,precision_cv_mean,precision_cv_std,recall_cv_mean,recall_cv_std,roc_auc_cv_mean,roc_auc_cv_std,time_seconds,best_params,model_path
0,XGBoost,base,0.941854,0.003506,0.941711,0.003435,0.944133,0.004759,0.939306,0.002331,0.987294,0.001545,13.58,,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...
1,LightGBM,base,0.941855,0.003398,0.941691,0.003532,0.944237,0.003085,0.939184,0.006175,0.987132,0.001651,12.15,,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...


## Tuning de XGBoost y LightGBM

Se mantiene F1 como metrica de busqueda para equilibrar precision y recall en `Revenue=True`. La decision final se valida aparte en test real.


In [16]:
xgb_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
)

param_dist_xgb = {
    "n_estimators": randint(100, 500),
    "max_depth": randint(3, 10),
    "learning_rate": loguniform(0.01, 0.3),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_weight": randint(1, 10),
    "gamma": uniform(0, 0.5),
}

random_xgb = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist_xgb,
    n_iter=20,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=42,
    return_train_score=True,
)

random_xgb.fit(X_train, y_train)
print("Mejor F1 CV XGBoost:", random_xgb.best_score_)
print("Mejores parametros XGBoost:")
print(random_xgb.best_params_)


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Mejor F1 CV XGBoost: 0.9449587156275949
Mejores parametros XGBoost:
{'colsample_bytree': np.float64(0.7297380084021096), 'gamma': np.float64(0.061043977350336676), 'learning_rate': np.float64(0.03359658305322321), 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 484, 'subsample': np.float64(0.6911740650167767)}


In [17]:
best_xgb = random_xgb.best_estimator_

xgb_tuned_result = evaluate_model_cv(
    model=best_xgb,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    model_name="XGBoost",
    version="tuned",
)

best_xgb.fit(X_train, y_train)
joblib.dump(best_xgb, XGB_TUNED_PATH)
trained_models[("XGBoost", "tuned")] = best_xgb
results.append({
    **xgb_tuned_result,
    "best_params": str(random_xgb.best_params_),
    "model_path": str(XGB_TUNED_PATH),
})

pd.DataFrame(results).sort_values("f1_cv_mean", ascending=False)


,model,version,accuracy_cv_mean,accuracy_cv_std,f1_cv_mean,f1_cv_std,precision_cv_mean,precision_cv_std,recall_cv_mean,recall_cv_std,roc_auc_cv_mean,roc_auc_cv_std,time_seconds,best_params,model_path
2,XGBoost,tuned,0.944950,0.002039,0.944959,0.002075,0.944789,0.001759,0.945133,0.003170,0.988469,0.001302,6.01,{'colsample_bytree': np.float64(0.729738008402...,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...
0,XGBoost,base,0.941854,0.003506,0.941711,0.003435,0.944133,0.004759,0.939306,0.002331,0.987294,0.001545,13.58,,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...
1,LightGBM,base,0.941855,0.003398,0.941691,0.003532,0.944237,0.003085,0.939184,0.006175,0.987132,0.001651,12.15,,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...


In [18]:
if LIGHTGBM_AVAILABLE:
    lgbm_model = LGBMClassifier(
        random_state=42,
        verbose=-1,
        n_jobs=-1,
    )

    param_dist_lgbm = {
        "n_estimators": randint(100, 500),
        "max_depth": randint(3, 12),
        "learning_rate": loguniform(0.01, 0.3),
        "num_leaves": randint(15, 80),
        "subsample": uniform(0.6, 0.4),
        "colsample_bytree": uniform(0.6, 0.4),
        "min_child_samples": randint(10, 80),
        "reg_alpha": loguniform(1e-4, 1.0),
        "reg_lambda": loguniform(1e-4, 1.0),
    }

    random_lgbm = RandomizedSearchCV(
        estimator=lgbm_model,
        param_distributions=param_dist_lgbm,
        n_iter=20,
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        verbose=1,
        random_state=42,
        return_train_score=True,
    )

    random_lgbm.fit(X_train, y_train)
    print("Mejor F1 CV LightGBM:", random_lgbm.best_score_)
    print("Mejores parametros LightGBM:")
    print(random_lgbm.best_params_)
else:
    random_lgbm = None
    print("LightGBM no esta instalado en este entorno. Se omite tuning LightGBM.")


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Mejor F1 CV LightGBM: 0.9427976055110328
Mejores parametros LightGBM:
{'colsample_bytree': np.float64(0.8733054075301833), 'learning_rate': np.float64(0.07962309016748462), 'max_depth': 10, 'min_child_samples': 56, 'n_estimators': 134, 'num_leaves': 50, 'reg_alpha': np.float64(0.000535728006960183), 'reg_lambda': np.float64(0.10506199451597038), 'subsample': np.float64(0.7700623497964979)}


In [19]:
if LIGHTGBM_AVAILABLE and random_lgbm is not None:
    best_lgbm = random_lgbm.best_estimator_

    lgbm_tuned_result = evaluate_model_cv(
        model=best_lgbm,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        model_name="LightGBM",
        version="tuned",
    )

    best_lgbm.fit(X_train, y_train)
    joblib.dump(best_lgbm, LGBM_TUNED_PATH)
    trained_models[("LightGBM", "tuned")] = best_lgbm
    results.append({
        **lgbm_tuned_result,
        "best_params": str(random_lgbm.best_params_),
        "model_path": str(LGBM_TUNED_PATH),
    })

pd.DataFrame(results).sort_values("f1_cv_mean", ascending=False)


,model,version,accuracy_cv_mean,accuracy_cv_std,f1_cv_mean,f1_cv_std,precision_cv_mean,precision_cv_std,recall_cv_mean,recall_cv_std,roc_auc_cv_mean,roc_auc_cv_std,time_seconds,best_params,model_path
2,XGBoost,tuned,0.944950,0.002039,0.944959,0.002075,0.944789,0.001759,0.945133,0.003170,0.988469,0.001302,6.01,{'colsample_bytree': np.float64(0.729738008402...,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...
3,LightGBM,tuned,0.942765,0.003191,0.942798,0.003166,0.942292,0.004011,0.943312,0.003376,0.987720,0.001383,2.43,{'colsample_bytree': np.float64(0.873305407530...,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...
0,XGBoost,base,0.941854,0.003506,0.941711,0.003435,0.944133,0.004759,0.939306,0.002331,0.987294,0.001545,13.58,,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...
1,LightGBM,base,0.941855,0.003398,0.941691,0.003532,0.944237,0.003085,0.939184,0.006175,0.987132,0.001651,12.15,,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...


## Evaluacion en test real

Esta es la tabla clave para comparar contra Random Forest: todos los modelos se evaluan sobre `test_processed.csv`, que conserva la proporcion real de compras cercana al 15%.


In [20]:
if RF_FINAL_PATH.exists():
    rf_final = joblib.load(RF_FINAL_PATH)
    assert_feature_compatibility(rf_final, X_test, "Random Forest final")
    trained_models[("Random Forest", "final_test")] = rf_final
else:
    print("No se encontro Random Forest final en:", RF_FINAL_PATH)

test_rows = []

for (model_name, version), model in trained_models.items():
    assert_feature_compatibility(model, X_test, f"{model_name} {version}")
    for threshold in THRESHOLDS:
        row = {
            "model": model_name,
            "version": version,
            **evaluate_model_test(model, X_test, y_test, threshold=threshold),
        }
        test_rows.append(row)

test_results_df = pd.DataFrame(test_rows).sort_values(
    ["threshold", "test_f1"],
    ascending=[False, False],
).reset_index(drop=True)

display(test_results_df)


,model,version,threshold,test_accuracy,test_f1,test_precision,test_recall,test_roc_auc,test_pr_auc,test_predicted_positive,test_tn,test_fp,test_fn,test_tp
0,Random Forest,final_test,0.50,0.895125,0.675127,0.655172,0.696335,0.921000,0.694010,406,1919,140,116,266
1,XGBoost,tuned,0.50,0.899631,0.672021,0.687671,0.657068,0.931870,0.743839,365,1945,114,131,251
2,LightGBM,base,0.50,0.897993,0.668442,0.680217,0.657068,0.931654,0.736090,369,1941,118,131,251
3,LightGBM,tuned,0.50,0.898402,0.667560,0.684066,0.651832,0.931540,0.741170,364,1944,115,133,249
4,XGBoost,base,0.50,0.896764,0.664894,0.675676,0.654450,0.930952,0.736823,370,1939,120,132,250
5,LightGBM,base,0.25,0.873822,0.666667,0.568266,0.806283,0.931654,0.736090,542,1825,234,74,308
6,XGBoost,base,0.25,0.874642,0.663736,0.571970,0.790576,0.930952,0.736823,528,1833,226,80,302
7,XGBoost,tuned,0.25,0.875051,0.663727,0.573333,0.787958,0.931870,0.743839,525,1835,224,81,301
8,LightGBM,tuned,0.25,0.869726,0.658798,0.558182,0.803665,0.931540,0.741170,550,1816,243,75,307
9,Random Forest,final_test,0.25,0.843097,0.636277,0.499255,0.876963,0.921000,0.694010,671,1723,336,47,335


In [21]:
cv_results_df = pd.DataFrame(results)

if not cv_results_df.empty:
    cv_results_df = cv_results_df.sort_values(
        "f1_cv_mean",
        ascending=False,
    ).reset_index(drop=True)

summary_rows = []
for _, test_row in test_results_df.iterrows():
    cv_match = cv_results_df[
        (cv_results_df["model"] == test_row["model"]) &
        (cv_results_df["version"] == test_row["version"])
    ]

    base = test_row.to_dict()
    if not cv_match.empty:
        cv_data = cv_match.iloc[0].to_dict()
        for key, value in cv_data.items():
            if key not in base:
                base[key] = value
    summary_rows.append(base)

final_results_df = pd.DataFrame(summary_rows)
final_results_df.to_csv(RESULTS_PATH, index=False)

print("Resultados guardados en:", RESULTS_PATH)
display(final_results_df)


Resultados guardados en: D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\models\ensemble_results_xgb_lgbm.csv


,model,version,threshold,test_accuracy,test_f1,test_precision,test_recall,test_roc_auc,test_pr_auc,test_predicted_positive,...,f1_cv_std,precision_cv_mean,precision_cv_std,recall_cv_mean,recall_cv_std,roc_auc_cv_mean,roc_auc_cv_std,time_seconds,best_params,model_path
0,Random Forest,final_test,0.50,0.895125,0.675127,0.655172,0.696335,0.921000,0.694010,406,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,XGBoost,tuned,0.50,0.899631,0.672021,0.687671,0.657068,0.931870,0.743839,365,...,0.002075,0.944789,0.001759,0.945133,0.003170,0.988469,0.001302,6.01,{'colsample_bytree': np.float64(0.729738008402...,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...
2,LightGBM,base,0.50,0.897993,0.668442,0.680217,0.657068,0.931654,0.736090,369,...,0.003532,0.944237,0.003085,0.939184,0.006175,0.987132,0.001651,12.15,,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...
3,LightGBM,tuned,0.50,0.898402,0.667560,0.684066,0.651832,0.931540,0.741170,364,...,0.003166,0.942292,0.004011,0.943312,0.003376,0.987720,0.001383,2.43,{'colsample_bytree': np.float64(0.873305407530...,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...
4,XGBoost,base,0.50,0.896764,0.664894,0.675676,0.654450,0.930952,0.736823,370,...,0.003435,0.944133,0.004759,0.939306,0.002331,0.987294,0.001545,13.58,,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...
5,LightGBM,base,0.25,0.873822,0.666667,0.568266,0.806283,0.931654,0.736090,542,...,0.003532,0.944237,0.003085,0.939184,0.006175,0.987132,0.001651,12.15,,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...
6,XGBoost,base,0.25,0.874642,0.663736,0.571970,0.790576,0.930952,0.736823,528,...,0.003435,0.944133,0.004759,0.939306,0.002331,0.987294,0.001545,13.58,,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...
7,XGBoost,tuned,0.25,0.875051,0.663727,0.573333,0.787958,0.931870,0.743839,525,...,0.002075,0.944789,0.001759,0.945133,0.003170,0.988469,0.001302,6.01,{'colsample_bytree': np.float64(0.729738008402...,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...
8,LightGBM,tuned,0.25,0.869726,0.658798,0.558182,0.803665,0.931540,0.741170,550,...,0.003166,0.942292,0.004011,0.943312,0.003376,0.987720,0.001383,2.43,{'colsample_bytree': np.float64(0.873305407530...,D:\_Trabajos\Desktop\Proyecto AD\dp261-g4\mode...
9,Random Forest,final_test,0.25,0.843097,0.636277,0.499255,0.876963,0.921000,0.694010,671,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Lectura de negocio

- `threshold=0.50`: escenario conservador, prioriza precision.
- `threshold=0.25`: escenario orientado a capturar mas compradores, prioriza recall.
- La seleccion final no debe hacerse solo por F1 CV; debe mirar el test real y el costo/beneficio de contactar usuarios.


In [22]:
if not test_results_df.empty:
    business_cols = [
        "model",
        "version",
        "threshold",
        "test_precision",
        "test_recall",
        "test_f1",
        "test_roc_auc",
        "test_pr_auc",
        "test_predicted_positive",
        "test_tp",
        "test_fp",
        "test_fn",
    ]
    display(test_results_df[business_cols].sort_values(["threshold", "test_recall"], ascending=[False, False]))


,model,version,threshold,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc,test_predicted_positive,test_tp,test_fp,test_fn
0,Random Forest,final_test,0.50,0.655172,0.696335,0.675127,0.921000,0.694010,406,266,140,116
1,XGBoost,tuned,0.50,0.687671,0.657068,0.672021,0.931870,0.743839,365,251,114,131
2,LightGBM,base,0.50,0.680217,0.657068,0.668442,0.931654,0.736090,369,251,118,131
4,XGBoost,base,0.50,0.675676,0.654450,0.664894,0.930952,0.736823,370,250,120,132
3,LightGBM,tuned,0.50,0.684066,0.651832,0.667560,0.931540,0.741170,364,249,115,133
9,Random Forest,final_test,0.25,0.499255,0.876963,0.636277,0.921000,0.694010,671,335,336,47
5,LightGBM,base,0.25,0.568266,0.806283,0.666667,0.931654,0.736090,542,308,234,74
8,LightGBM,tuned,0.25,0.558182,0.803665,0.658798,0.931540,0.741170,550,307,243,75
6,XGBoost,base,0.25,0.571970,0.790576,0.663736,0.930952,0.736823,528,302,226,80
7,XGBoost,tuned,0.25,0.573333,0.787958,0.663727,0.931870,0.743839,525,301,224,81


## Evaluacion por VisitorType

Esta seccion separa el test real por tipo de visitante. La lectura de negocio es distinta para usuarios nuevos y recurrentes: contactar nuevos suele ser mas caro, mientras que activar recurrentes suele tener menor costo.


In [ ]:
VISITOR_TYPE_COLS = [col for col in X_test.columns if col.startswith("cat__VisitorType_")]

if not VISITOR_TYPE_COLS:
    raise ValueError("No se encontraron columnas one-hot de VisitorType en X_test.")

visitor_type_counts = X_test[VISITOR_TYPE_COLS].sum().sort_values(ascending=False)
print("Columnas VisitorType encontradas:")
print(visitor_type_counts)


def get_visitor_segment_masks(X):
    masks = {}
    for col in VISITOR_TYPE_COLS:
        segment_name = col.replace("cat__VisitorType_", "")
        masks[segment_name] = X[col] == 1
    return masks

visitor_segment_masks = get_visitor_segment_masks(X_test)


In [ ]:
def safe_binary_metrics(y_true, y_pred, y_proba):
    positives = int(y_true.sum())
    negatives = int((y_true == 0).sum())

    metrics = {
        "segment_size": int(len(y_true)),
        "segment_buyers": positives,
        "segment_non_buyers": negatives,
        "segment_conversion_rate": float(y_true.mean()) if len(y_true) else np.nan,
        "accuracy": accuracy_score(y_true, y_pred) if len(y_true) else np.nan,
        "precision": precision_score(y_true, y_pred, zero_division=0) if len(y_true) else np.nan,
        "recall": recall_score(y_true, y_pred, zero_division=0) if positives > 0 else np.nan,
        "f1": f1_score(y_true, y_pred, zero_division=0) if len(y_true) else np.nan,
        "predicted_positive": int(y_pred.sum()),
        "tp": int(((y_true == 1) & (y_pred == 1)).sum()),
        "fp": int(((y_true == 0) & (y_pred == 1)).sum()),
        "fn": int(((y_true == 1) & (y_pred == 0)).sum()),
        "tn": int(((y_true == 0) & (y_pred == 0)).sum()),
    }

    if positives > 0 and negatives > 0:
        metrics["roc_auc"] = roc_auc_score(y_true, y_proba)
        metrics["pr_auc"] = average_precision_score(y_true, y_proba)
    else:
        metrics["roc_auc"] = np.nan
        metrics["pr_auc"] = np.nan

    return metrics


segment_rows = []

for (model_name, version), model in trained_models.items():
    y_proba_full = model.predict_proba(X_test)[:, 1]

    for segment_name, mask in visitor_segment_masks.items():
        if mask.sum() == 0:
            continue

        y_segment = y_test.loc[mask].astype(int)
        proba_segment = y_proba_full[mask.to_numpy()]

        for threshold in THRESHOLDS:
            pred_segment = (proba_segment >= threshold).astype(int)
            row = {
                "model": model_name,
                "version": version,
                "visitor_type": segment_name,
                "threshold": threshold,
                **safe_binary_metrics(y_segment, pred_segment, proba_segment),
            }
            segment_rows.append(row)

segment_results_df = pd.DataFrame(segment_rows).sort_values(
    ["visitor_type", "threshold", "f1"],
    ascending=[True, False, False],
).reset_index(drop=True)

display(segment_results_df)


In [ ]:
SEGMENT_RESULTS_PATH = MODELS_DIR / "ensemble_results_by_visitor_type.csv"
segment_results_df.to_csv(SEGMENT_RESULTS_PATH, index=False)

print("Resultados segmentados guardados en:", SEGMENT_RESULTS_PATH)

segment_business_cols = [
    "visitor_type",
    "model",
    "version",
    "threshold",
    "segment_size",
    "segment_buyers",
    "segment_conversion_rate",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "predicted_positive",
    "tp",
    "fp",
    "fn",
]

display(segment_results_df[segment_business_cols])


## Recomendacion segmentada

- Para `New_Visitor`, conviene priorizar precision y PR-AUC porque el contacto suele ser mas caro.
- Para `Returning_Visitor`, puede aceptarse un threshold mas bajo si el costo de activacion es menor y se busca capturar mas compradores.
- `Other` tiene muy pocos casos en test, por lo que sus metricas deben leerse solo como referencia.


In [ ]:
def pick_segment_recommendations(segment_df):
    recommendations = []

    for segment_name, df_seg in segment_df.groupby("visitor_type"):
        if segment_name == "New_Visitor":
            candidates = df_seg[df_seg["threshold"] >= 0.50].copy()
            if candidates.empty:
                candidates = df_seg.copy()
            selected = candidates.sort_values(
                ["precision", "pr_auc", "f1"],
                ascending=False,
            ).iloc[0]
            objective = "Priorizar precision por mayor costo de contacto"
        elif segment_name == "Returning_Visitor":
            candidates = df_seg[df_seg["threshold"] <= 0.25].copy()
            if candidates.empty:
                candidates = df_seg.copy()
            selected = candidates.sort_values(
                ["recall", "f1", "precision"],
                ascending=False,
            ).iloc[0]
            objective = "Priorizar recall por menor costo de activacion"
        else:
            candidates = df_seg[df_seg["threshold"] >= 0.50].copy()
            if candidates.empty:
                candidates = df_seg.copy()
            selected = candidates.sort_values(
                ["precision", "f1"],
                ascending=False,
            ).iloc[0]
            objective = "Politica conservadora por bajo volumen"

        row = selected.to_dict()
        row["recommended_objective"] = objective
        recommendations.append(row)

    return pd.DataFrame(recommendations)


segment_recommendations_df = pick_segment_recommendations(segment_results_df)

display(segment_recommendations_df[[
    "visitor_type",
    "model",
    "version",
    "threshold",
    "recommended_objective",
    "segment_size",
    "segment_buyers",
    "precision",
    "recall",
    "f1",
    "pr_auc",
    "predicted_positive",
    "tp",
    "fp",
    "fn",
]])
